# 8-8 End-to-End MLP — Advanced Practice

강의 원문 대신 직접 작성한 코드, 실행 결과와 학습 메모를 정리했습니다.


In [1]:
# 검증 가능 정답 코드
# snapshot의 결함을 데이터 계약, train gradient 경계, validation 불변성 순으로 분류합니다.
snapshot = {
    "y_dtype": "float32",
    "train_zero_each_batch": False,
    "valid_no_grad": False,
    "valid_backward": True,
    "valid_step": True,
}
violations = []
if snapshot["y_dtype"] != "int64": violations.append("target_dtype")
if not snapshot["train_zero_each_batch"]: violations.append("gradient_reset")
if not snapshot["valid_no_grad"]: violations.append("validation_grad_tracking")
if snapshot["valid_backward"]: violations.append("validation_backward")
if snapshot["valid_step"]: violations.append("validation_update")
# 각 수정 뒤 확인할 dtype·독립 gradient·검증 weight 불변 조건을 결과에 함께 남깁니다.
print(f"violations={violations}")
print("fix_order=data -> train_step -> validation_step")
print("postcheck=long target, independent gradients, unchanged validation weights")

violations=['target_dtype', 'gradient_reset', 'validation_grad_tracking', 'validation_backward', 'validation_update']
fix_order=data -> train_step -> validation_step
postcheck=long target, independent gradients, unchanged validation weights


In [2]:
# 검증 가능 정답 코드
# 공통 epoch 함수에서도 training flag에 따라 gradient 활성화와 optimizer 호출 범위를 명확히 나눕니다.
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(12)
train_x = torch.tensor([[2.,0.,1.,0.],[0.,2.,0.,1.],[2.,1.,1.,0.],[0.,1.,0.,2.],[-2.,0.,-1.,0.],[0.,-2.,0.,-1.],[-2.,-1.,-1.,0.],[0.,-1.,0.,-2.]])
train_y = torch.tensor([1,1,1,1,0,0,0,0], dtype=torch.long)
valid_x = torch.tensor([[1.,0.,1.,0.],[0.,1.,0.,1.],[-1.,0.,-1.,0.],[0.,-1.,0.,-1.]])
valid_y = torch.tensor([1,1,0,0], dtype=torch.long)
train_loader = DataLoader(TensorDataset(train_x, train_y), batch_size=4, shuffle=False)
valid_loader = DataLoader(TensorDataset(valid_x, valid_y), batch_size=3, shuffle=False)
model = nn.Sequential(nn.Linear(4, 6), nn.ReLU(), nn.Linear(6, 2))
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.2)

def run_epoch(training, loader):
    model.train(training)
    total_loss = seen = 0
    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for x, y in loader:
            if training: optimizer.zero_grad()
            loss = criterion(model(x), y)
            if training:
                loss.backward(); optimizer.step()
            total_loss += loss.item() * y.shape[0]
            seen += y.shape[0]
    return total_loss / seen

history = {"train_loss": [], "valid_loss": []}
valid_unchanged = True
for _ in range(3):
    history["train_loss"].append(run_epoch(True, train_loader))
    before = [p.detach().clone() for p in model.parameters()]
    history["valid_loss"].append(run_epoch(False, valid_loader))
    valid_unchanged &= all(torch.equal(a, b) for a, b in zip(before, model.parameters()))
# 세 epoch의 train 감소와 매 validation 전후 state 일치를 별도 boolean으로 검증합니다.
print(f"epochs={len(history['train_loss'])}")
print(f"train_loss_decreased={history['train_loss'][-1] < history['train_loss'][0]}") #로스줄었는지.
print(f"validation_weights_unchanged={valid_unchanged}")

epochs=3
train_loss_decreased=True
validation_weights_unchanged=True


In [3]:
# 검증 가능 정답 코드
# config와 validation 기반 best epoch를 먼저 확정하고 test 결과와 평가 횟수는 그 뒤 report에 붙입니다.
valid_loss = [0.80, 0.62, 0.67]
best_index = min(range(len(valid_loss)), key=valid_loss.__getitem__)
report = {
    "config": {"input_dim": 4, "hidden_dim": 6, "classes": 2, "epochs": 3},
    "best_epoch": best_index + 1,
    "best_valid_loss": valid_loss[best_index],
    "final_valid_loss": valid_loss[-1],
    "test_accuracy": 0.78,
    "test_evaluations": 1,
}
# test를 한 번만 최종 확인에 썼다는 decision_basis를 출력해 재튜닝 자료와 분리합니다.
print(report)
print("decision_basis=validation; test=final check only")

{'config': {'input_dim': 4, 'hidden_dim': 6, 'classes': 2, 'epochs': 3}, 'best_epoch': 2, 'best_valid_loss': 0.62, 'final_valid_loss': 0.67, 'test_accuracy': 0.78, 'test_evaluations': 1}
decision_basis=validation; test=final check only
